# ACLED Source-Split EDA (National vs Regional/Local Newspapers)

This notebook does **exactly** what Nausheen asked:

1. **Split ACLED events by reporting source** into:
   - **National-level newspapers**
   - **Regional/Local newspapers**
2. Run **basic EDA** comparing the two subsets:
   - event counts
   - fatalities
   - event-type composition
   - geographic coverage (states/districts/locations)
   - state-level skew (where national vs local dominates)
   - trends over time (events & fatalities by year)
   - severity proxy (fatalities per 100 events by event type)
3. **Exports everything** (tables + plots + classified dataset) into an `outputs/` folder and creates a ZIP.

---

## What you must customize (1 minute)

- **Edit `NATIONAL_NEWSPAPERS`** to match your team’s definition of *national-level newspapers*.
- Everything not in `NATIONAL_NEWSPAPERS` and not in `NON_NEWSPAPER_SOURCES` is treated as **regional/local newspaper**.

> ACLED events can cite multiple sources. We therefore classify at the event level by scanning all sources listed in the `source` column.


In [ ]:
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# CONFIG (EDIT ONLY THESE)
# ----------------------------

# Path to your ACLED CSV (India extract)
ACLED_CSV_PATH = "ACLED Data_2025-12-27.csv"   # <-- change if needed

# Date window for analysis (matches your 2016–2024 comparison window)
DATE_MIN = "2016-01-01"
DATE_MAX = "2024-12-31"

# Output folder (all exports go here)
OUTDIR = Path("outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) DEFINE NATIONAL NEWSPAPERS
# ----------------------------
# Put ONLY the sources you want classified as "national-level newspapers".
# Names should match ACLED's 'source' strings exactly (case/spaces usually match well).
NATIONAL_NEWSPAPERS = {
    # Common national papers (edit as needed)
    "Times of India",
    "The Hindu",
    "Hindustan Times",
    "Indian Express",
    "The Indian Express",
    "The New Indian Express",
    "The Tribune",
    "Telegraph (India)",
    "Business Standard (India)",
    "Economic Times (India)",
    "Live Mint",
    "Outlook",
    "Outlook (India)",
    "The Print",
    "Week (India)",
    "Statesman (India)",
}

# ----------------------------
# 2) DEFINE NON-NEWSPAPER SOURCES (OPTIONAL BUT RECOMMENDED)
# ----------------------------
# These are agencies/NGOs/broadcasters/international media etc.
# We keep them separate so they do NOT get incorrectly labeled as "regional/local newspapers".
NON_NEWSPAPER_SOURCES = {
    # Wires / agencies
    "Press Trust of India",
    "United News of India",
    "Indo-Asian News Service",
    "Asia News International (ANI)",
    "Asian News International",
    "Reuters",
    "AFP (Agence France-Presse)",
    "AP (Associated Press)",

    # International / broadcasters
    "BBC News",
    "Deutsche Welle",
    "Xinhua",
    "All India Radio",
    "Al Jazeera",

    # NGOs / orgs / databases
    "Amnesty International",
    "CIVICUS (NGO)",
    "Committee to Protect Journalists (NGO)",
    "Commonwealth Human Rights Initiative (NGO)",
    "Forum Asia (NGO)",
    "Front Line Defenders (NGO)",
    "Global Witness (NGO)",
    "Scholars at Risk (NGO)",
    "Aid Worker Security Database",
    "Insecurity Insight",
    "ProtectDefenders.eu",

    # Other portals
    "GardaWorld",
    "South Asia Terrorism Portal",
}

def _norm(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

NATIONAL_NEWSPAPERS = {_norm(x) for x in NATIONAL_NEWSPAPERS}
NON_NEWSPAPER_SOURCES = {_norm(x) for x in NON_NEWSPAPER_SOURCES}

print("National newspapers:", len(NATIONAL_NEWSPAPERS))
print("Non-newspaper sources:", len(NON_NEWSPAPER_SOURCES))


In [ ]:
# ----------------------------
# Load ACLED
# ----------------------------
df = pd.read_csv(ACLED_CSV_PATH)

required = {"event_date","source","event_type","fatalities","admin1","admin2","location"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df = df[(df["event_date"] >= pd.to_datetime(DATE_MIN)) & (df["event_date"] <= pd.to_datetime(DATE_MAX))].copy()
df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").fillna(0)

print("Rows:", len(df))
print("Date range:", df["event_date"].min(), "->", df["event_date"].max())
print("Total fatalities:", int(df["fatalities"].sum()))


## Classification logic (exact)

ACLED `source` can contain **multiple sources** for one event (often separated by `;` and sometimes commas).

For each event, we split the `source` string into a list of sources and compute:

- `has_national`: any source in `NATIONAL_NEWSPAPERS`
- `has_regional`: any source that is **not** in `NATIONAL_NEWSPAPERS` and **not** in `NON_NEWSPAPER_SOURCES`
- `has_non_news`: any source in `NON_NEWSPAPER_SOURCES`

We assign a **4-way bucket**:

- `mixed_nat+regional` (both national and regional/local present)
- `national_newspaper` (national present, no regional/local)
- `regional_local_newspaper` (regional/local present, no national)
- `no_newspaper_match` (only non-newspaper sources, or empty)

Then we create the **2-way split requested by Nausheen**:

- `national_incl_mixed` = `national_newspaper` + `mixed_nat+regional`
- `regional_local` = `regional_local_newspaper`

`no_newspaper_match` is kept separate as `other` (not used in the 2-way comparison).


In [ ]:
def tokenize_sources(source_str: str):
    """Split ACLED source string into individual sources."""
    if pd.isna(source_str):
        return []
    s = str(source_str)
    parts = []
    for chunk in s.split(";"):
        for sub in chunk.split(","):
            t = sub.strip()
            if t:
                parts.append(_norm(t))
    return parts

def classify_bucket(source_str: str):
    sources = tokenize_sources(source_str)
    if not sources:
        return {"has_national": False, "has_regional": False, "has_non_news": False, "bucket_4way": "no_newspaper_match"}

    has_nat = any(s in NATIONAL_NEWSPAPERS for s in sources)
    has_non = any(s in NON_NEWSPAPER_SOURCES for s in sources)
    has_reg = any((s not in NATIONAL_NEWSPAPERS) and (s not in NON_NEWSPAPER_SOURCES) for s in sources)

    if has_nat and has_reg:
        b = "mixed_nat+regional"
    elif has_nat:
        b = "national_newspaper"
    elif has_reg:
        b = "regional_local_newspaper"
    else:
        b = "no_newspaper_match"

    return {"has_national": has_nat, "has_regional": has_reg, "has_non_news": has_non, "bucket_4way": b}

cls = df["source"].apply(classify_bucket).apply(pd.Series)
df = pd.concat([df, cls], axis=1)

df["bucket_2way"] = np.where(
    df["bucket_4way"].isin(["national_newspaper", "mixed_nat+regional"]),
    "national_incl_mixed",
    np.where(df["bucket_4way"].eq("regional_local_newspaper"), "regional_local", "other")
)

df["bucket_4way"].value_counts(), df["bucket_2way"].value_counts()


## EDA outputs generated

This section creates and exports:

### Tables (CSV)
- `acled_with_source_buckets_2016_2024.csv` (full classified dataset)
- `acled_newspaper_bucket_summary_4way.csv`
- `acled_newspaper_bucket_summary_2way.csv`
- `acled_event_type_shares_2way.csv`
- `acled_state_skew_2way.csv`
- `acled_yearly_events_fatalities_2way.csv`
- `acled_severity_by_event_type_2way.csv`

### Plots (PNG)
- events and fatalities by subset (2-way)
- event-type composition comparison (2-way)
- state skew (top 15 national-heavy and top 15 regional-heavy)
- yearly events and yearly fatalities (2-way)

Finally it zips everything into `acled_source_split_outputs.zip`.


In [ ]:
# ----------------------------
# Tables
# ----------------------------
def summary_table(data, group_col):
    out = (data.groupby(group_col)
             .agg(
                 events=("event_date", "size"),
                 fatalities=("fatalities", "sum"),
                 mean_fatalities=("fatalities", "mean"),
                 median_fatalities=("fatalities", "median"),
                 states=("admin1", "nunique"),
                 districts=("admin2", "nunique"),
                 locations=("location", "nunique"),
             )
             .reset_index()
             .sort_values("events", ascending=False))
    out["fatalities"] = out["fatalities"].astype(int)
    return out

summary_4way = summary_table(df, "bucket_4way")
summary_2way = summary_table(df[df["bucket_2way"].isin(["national_incl_mixed","regional_local"])], "bucket_2way")

summary_4way.to_csv(OUTDIR/"acled_newspaper_bucket_summary_4way.csv", index=False)
summary_2way.to_csv(OUTDIR/"acled_newspaper_bucket_summary_2way.csv", index=False)

# Event-type composition (2-way)
sub = df[df["bucket_2way"].isin(["national_incl_mixed","regional_local"])].copy()
event_type_shares = (pd.crosstab(sub["event_type"], sub["bucket_2way"], normalize="columns") * 100).round(2)
event_type_shares.to_csv(OUTDIR/"acled_event_type_shares_2way.csv")

# State-level skew (2-way): share(national) - share(regional)
state_share = pd.crosstab(sub["admin1"], sub["bucket_2way"], normalize="columns")
state_share["diff_nat_minus_reg"] = state_share["national_incl_mixed"] - state_share["regional_local"]
state_share_sorted = state_share.sort_values("diff_nat_minus_reg", ascending=False)
state_share_sorted.reset_index().to_csv(OUTDIR/"acled_state_skew_2way.csv", index=False)

# Trends by year
sub["year"] = sub["event_date"].dt.year
yearly = (sub.groupby(["year","bucket_2way"])
            .agg(events=("event_date","size"), fatalities=("fatalities","sum"))
            .reset_index())
yearly.to_csv(OUTDIR/"acled_yearly_events_fatalities_2way.csv", index=False)

# Severity proxy: fatalities per 100 events by event type & subset
sev = (sub.groupby(["bucket_2way","event_type"])
         .agg(events=("event_date","size"), fatalities=("fatalities","sum"))
         .reset_index())
sev["fatalities_per_100_events"] = (sev["fatalities"] / sev["events"] * 100).replace([np.inf, -np.inf], np.nan)
sev.sort_values("fatalities_per_100_events", ascending=False).to_csv(
    OUTDIR/"acled_severity_by_event_type_2way.csv", index=False
)

# Save classified dataset
df.to_csv(OUTDIR/"acled_with_source_buckets_2016_2024.csv", index=False)

# ----------------------------
# Plots (Matplotlib default colors)
# ----------------------------
def bar_plot(x, y, title, xlabel, ylabel, fname, rotate=0):
    plt.figure(figsize=(9,5))
    plt.bar(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if rotate:
        plt.xticks(rotation=rotate, ha="right")
    plt.tight_layout()
    plt.savefig(OUTDIR/fname, dpi=200)
    plt.close()

# Events + fatalities by subset
tmp = summary_2way.set_index("bucket_2way")
bar_plot(tmp.index.tolist(), tmp["events"].tolist(),
         "ACLED: Events by reporting subset (2-way)",
         "Subset", "Events", "eda_events_by_subset_2way.png")

bar_plot(tmp.index.tolist(), tmp["fatalities"].tolist(),
         "ACLED: Fatalities by reporting subset (2-way)",
         "Subset", "Fatalities", "eda_fatalities_by_subset_2way.png")

# Event-type composition side-by-side bars
etype = event_type_shares.copy()
etype = etype.loc[etype.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(10,6))
x = np.arange(len(etype.index))
w = 0.4
plt.bar(x - w/2, etype["national_incl_mixed"], width=w, label="national_incl_mixed")
plt.bar(x + w/2, etype["regional_local"], width=w, label="regional_local")
plt.xticks(x, etype.index, rotation=30, ha="right")
plt.ylabel("Share of events (%)")
plt.title("ACLED: Event-type composition by subset (2-way)")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR/"eda_event_type_composition_2way.png", dpi=200)
plt.close()

# State skew plots: top 15 national-heavy and top 15 regional-heavy
diff = state_share_sorted["diff_nat_minus_reg"]
top_nat = diff.head(15)
top_reg = diff.tail(15)

plt.figure(figsize=(10,6))
plt.bar(top_nat.index, top_nat.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share difference (national - regional)")
plt.title("States more represented in national subset (top 15)")
plt.tight_layout()
plt.savefig(OUTDIR/"eda_state_skew_more_national.png", dpi=200)
plt.close()

plt.figure(figsize=(10,6))
plt.bar(top_reg.index, top_reg.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share difference (national - regional)")
plt.title("States more represented in regional/local subset (top 15)")
plt.tight_layout()
plt.savefig(OUTDIR/"eda_state_skew_more_regional.png", dpi=200)
plt.close()

# Yearly trend plots
pivot_events = yearly.pivot(index="year", columns="bucket_2way", values="events").fillna(0)
plt.figure(figsize=(10,6))
for col in pivot_events.columns:
    plt.plot(pivot_events.index, pivot_events[col], marker="o", label=col)
plt.title("ACLED: Events over time by subset (2-way)")
plt.xlabel("Year")
plt.ylabel("Events")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR/"eda_yearly_events_2way.png", dpi=200)
plt.close()

pivot_fat = yearly.pivot(index="year", columns="bucket_2way", values="fatalities").fillna(0)
plt.figure(figsize=(10,6))
for col in pivot_fat.columns:
    plt.plot(pivot_fat.index, pivot_fat[col], marker="o", label=col)
plt.title("ACLED: Fatalities over time by subset (2-way)")
plt.xlabel("Year")
plt.ylabel("Fatalities")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR/"eda_yearly_fatalities_2way.png", dpi=200)
plt.close()

# Manifest + ZIP
manifest = pd.DataFrame(
    [{"file": str(p), "bytes": p.stat().st_size} for p in sorted(OUTDIR.glob("**/*")) if p.is_file()]
)
manifest.to_csv(OUTDIR/"MANIFEST.csv", index=False)

zip_path = Path("acled_source_split_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTDIR.glob("**/*")):
        if p.is_file():
            z.write(p, arcname=str(p))

print("✅ Export complete.")
print("Outputs folder:", OUTDIR.resolve())
print("ZIP:", zip_path.resolve())
summary_2way


## Notes for verification

- If you change `NATIONAL_NEWSPAPERS`, re-run the notebook from the top.
- The 2-way comparison uses only `national_incl_mixed` and `regional_local`.
- `other` = events where sources are only agencies/NGOs/broadcasters etc. (no newspaper match under your definitions).
